# 📦 AI Inventory & Stock Count System — Google Colab Launcher

This notebook launches:
1. **Gradio Interactive Dashboard** (Inline Notebook UI + Public `gradio.live` link)
2. **Mobile UI & REST API** (Port `8765` exposed via Google Colab Native Proxy & Public Tunnels)

---

### Step 1: Install System & Python Dependencies
Installs `libzbar0` (required for barcode scanning in Linux) and all Python packages.

In [ ]:
# Install C-library for barcode scanning (pyzbar) and Python requirements
!apt-get update -qq && !apt-get install -y -qq libzbar0
!pip install -q -r requirements.txt pyngrok

### Step 2: (Optional) Configure ngrok Authtoken
If you have an ngrok authtoken, enter it below. If left blank, the launcher will automatically use **Google Colab Native Port Forwarding** (100% free, zero configuration needed).

In [ ]:
import os

# Optional: Set your ngrok authtoken (e.g. "2abc123...")
NGROK_AUTHTOKEN = ""

if NGROK_AUTHTOKEN.strip():
    os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN.strip()
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
        print("✅ ngrok authtoken configured.")
    except Exception as e:
        print("⚠️ Could not set ngrok authtoken:", e)

### Step 3: Launch Interactive Application & Generate Public Links

In [ ]:
import os
import sys
import time

# Set environment variables for Colab launch
os.environ["GRADIO_SHARE"] = "1"
os.environ["MOBILE_API_PORT"] = "8765"

print("=" * 70)
print("🚀 GENERATING PUBLIC MOBILE UI / REST API LINKS...")
print("=" * 70)

# 1. Native Google Colab Proxy Link
colab_proxy_url = None
try:
    from google.colab.output import eval_js
    colab_proxy_url = eval_js("google.colab.kernel.proxyPort(8765)")
    if colab_proxy_url:
        print("📱 Mobile UI (Colab Native Proxy) :", colab_proxy_url)
        print("📄 Mobile REST API Docs           :", colab_proxy_url + "docs")
        print("📊 API Stats Endpoint             :", colab_proxy_url + "api/stats")
except Exception:
    pass

# 2. ngrok Tunnel (if authtoken provided or set)
try:
    from pyngrok import ngrok
    mobile_tunnel = ngrok.connect(8765)
    print("🔗 Mobile UI (ngrok Public Link) :", mobile_tunnel.public_url)
    print("📄 Mobile REST API Docs (ngrok)   :", mobile_tunnel.public_url + "/docs")
except Exception as e:
    if not colab_proxy_url:
        print("💡 Note on Tunnel:", e)

print("=" * 70)
print("📊 Starting Gradio Dashboard & FastAPI Server...")
print("=" * 70)

# Import and launch main application inline
from app import main
main()